In [4]:
import requests
import json
import pandas as pd 
from datetime import datetime, date, time, timedelta
import calendar
import pytz

def get_token():
  request_url = "https://api.alertaq.com/api/v4/public/login"
  request_body = {
    "user": "prayagpurohit1@gmail.com",
    "password": "8238709119Pp!",
  }

  response = requests.post(request_url,json=request_body)

  login_response=json.loads(response.text)
  token = login_response['token']
  return token

def get_property_id(token, property_name):
    response = requests.get("https://api.alertaq.com/api/v4/public/locations", headers={"Token": token})
    locations = response.json()
    property_name = 'BGO Pen Center'
    matching_entries = [entry for entry in locations['dataModel'] if entry.get('name') == property_name]
    
    if matching_entries:
        return matching_entries[0]['_id']
    
    print("BGO Pen Center not found.")
    return None 

def get_sensorlist(token, bgo_id):
    # Get locations by id
    sensors_url = f"https://api.alertaq.com/api/v4/public/sensors?locationID={bgo_id}"
    response = requests.get(sensors_url, headers={"Token": token})
    sensors = json.loads(response.text)
    sensor_list_df = pd.DataFrame(sensors['dataModel'])
    # Filter only "Flowie-O" sensors
    flowie_sensors = sensor_list_df[sensor_list_df['friendlyType'] == "Flowie-O"]

    return flowie_sensors
    

def get_firstandlastdayofpreviousmonth():
    today = datetime.today()
    first_day_current_month = today.replace(day=1)
    last_day_previous_month = first_day_current_month - timedelta(days=1)
    first_day_previous_month = last_day_previous_month.replace(day=1)
    
    month_start = first_day_previous_month.strftime('%Y-%m-%d')
    month_end = last_day_previous_month.strftime('%Y-%m-%d')
    
    return month_start, month_end

def get_master_df():   
    month_start, month_end = get_firstandlastdayofpreviousmonth()
    master_df = pd.DataFrame()
    master_df['Date'] = pd.date_range(start=month_start, end=month_end)
    return master_df

In [ ]:
def get_timeseries_data(token, sensor_id, sensorstoquery): 
    rate = "d"
    series = "W"
    month_start, month_end = get_firstandlastdayofpreviousmonth()
    eastern = pytz.timezone('America/New_York')
    start_time_unix = int(eastern.localize(datetime.strptime(month_start, '%Y-%m-%d')).timestamp()) - 1000
    end_time_unix = int(eastern.localize(datetime.strptime(month_end, '%Y-%m-%d')).timestamp()) - 1000
    end_time_unix = int(datetime.strptime(month_end, '%Y-%m-%d').timestamp())
    timeseriesurl = f"https://api.alertaq.com/api/v4/public/timeseries?from={start_time_unix}&to={end_time_unix}&rate={rate}&series={series}&sensorID={sensor_id}"

    response = requests.get(timeseriesurl, headers={"Token": token})
    timeseries = json.loads(response.text)
    timeseries['dataModel'] = {
        sensorstoquery.get(sensor_id, sensor_id): data for sensor_id, data in timeseries['dataModel'].items()
    }
    sensor_id = list(timeseries['dataModel'].keys())[0]  # Assuming only one sensor ID
    # Get the corresponding sensor name or keep the ID if not found
    sensor_name = sensorstoquery.get(sensor_id, sensor_id)
    # Create the DataFrame
    timeseries_df = pd.DataFrame(timeseries['dataModel'][sensor_id], columns=['Date', f'{sensor_name}'])

    # Convert 'Date' column to datetime
    timeseries_df['Date'] = pd.to_datetime(timeseries_df['Date'], unit='ms')
    return timeseries_df

def mergewithmasterdf(master_df, timeseries_df):
    master_df = master_df.merge(timeseries_df, on='Date', how='left')
    return master_df

token = get_token()
bgo_id = get_property_id(token)
if bgo_id is None:
    raise ValueError("No valid location found. Exiting.")

sensor_list_df = get_sensorlist(token, bgo_id)
sensorstoquery = dict(zip(sensor_list_df['_id'], sensor_list_df['name']))  # Map sensor_id to name

master_df = get_master_df()
for sensor_id in sensorstoquery.keys():
    timeseries_df = get_timeseries_data(token, sensor_id, sensorstoquery)
    master_df = mergewithmasterdf(master_df, timeseries_df)



### Alert Lab main test

In [11]:
import Alertlab_api as atapi

token = atapi.get_token()
property_to_query = 'BGO Pen Center'
bgo_id = atapi.get_property_id(token, property_name=property_to_query)
if bgo_id is None:
    raise ValueError("No valid location found. Exiting.")

sensor_list_df = atapi.get_sensorlist(token, bgo_id)
sensorstoquery = dict(zip(sensor_list_df['_id'], sensor_list_df['name']))  # Map sensor_id to name

def mergewithmasterdf(master_df, timeseries_df):
    master_df = master_df.merge(timeseries_df, on='Date', how='left')
    return master_df

master_df = get_master_df()
for sensor_id in sensorstoquery.keys():
    timeseries_df = atapi.get_timeseries_data(token, sensor_id, sensorstoquery)
    master_df = mergewithmasterdf(master_df, timeseries_df)

master_df.head()

NameError: name 'timedelta' is not defined

In [9]:
import requests

def get_property_name_list(token):
    """
    Fetches the list of property names from the AlertAQ API, saves them into a text file,
    and returns the list of names.

    Parameters:
        token (str): The API token for authentication.

    Returns:
        list: A list of property names.
    """
    # API endpoint
    url = "https://api.alertaq.com/api/v4/public/locations"
    
    # Request headers
    headers = {"Token": token}
    
    try:
        # Fetch the response
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        # Parse the response JSON
        locations = response.json()
        
        # Extract the list of property names
        names = [location['name'] for location in locations['dataModel']]
        
        # Save the names into a text file
        with open("alert_lab_property_list.txt", "w") as file:
            for name in names:
                file.write(name + "\n")
        
        return names
    
    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching property names: {e}")
        return []

# Example usage:
# token = "your_api_token_here"
property_names = get_property_name_list(token)
print(property_names)


['Seneca', 'Huron', '455 Cochrane', 'Base Res', '475 Cochrane', 'Armel DHW Res', 'Norfinch', 'Armel Main Res', '130 Res', '39 Parkcrest Res', '45 Parkcrest Res', '440 Winona Res', '485 Patricia Res', 'Commodore A/B', 'Chelsea Shuttie', 'Stratford', 'Cavalier', 'Lasalle', 'Grenadier', 'Aventura 2', "Bob Langlois's Org", '35 Spencer Res', 'Chelsea', '125 Wellington Street North', 'BGO 10 Dundas Cooling Tower', 'BGO 10 Dundas Meter Room', 'PCC289 Res Whitehill', 'NHD Res 4001 Steeles Ave', 'PetroCan McCowan', 'Res BP Apartments 77 Davisville', 'PetroCan Major MacKenzie', 'PetroCan Brampton', 'Q Res - Branson Tower', 'PetroCan Pickering', 'Q Res - Kingside Apartments', 'Q Res - Mornelle Apartments', 'BGO 642 Dixon Swiss Chalet', 'Q Res - Res @ Y&C', 'BGO 620 Dixon Subway', 'BGO 620 Dixon  Tim Hortons', '321 Lakeshore Rd E', 'Res BP Apartments 55 Maitland', 'BGO Res Altair', 'Res BP Apartments 40 Alexander', 'BGO Res Durand', 'Res BP Apartments 50 Alexander', 'BGO 325 Central Parkway North'

## SimpleSUB API test